# 金铲铲之战 AI投放系统动力学仿真模型

## 问题定义

金铲铲之战为改善玩家体验投放大量AI进入游戏，但导致：
- **正向效应**：玩家更容易进入前四，体验更好，更愿意留存
- **负向效应**：段位膨胀，升段所需局数从140局降到70局，玩家更快达成目标后活跃度下降

**核心问题**：投放AI对玩家活跃到底是正向还是负向影响？有哪些调整方式可以优化？

In [ ]:
# ============================================================
# 第一部分：导入依赖
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'PingFang SC']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

print('依赖导入成功')

## 二、模型设计

### 2.1 玩家模型

每位玩家具有以下属性：
- **真实实力（Skill）**：服从 N(0,1) 分布，决定对局表现
- **当前段位（LP）**：从0开始，到达1000为大师
- **动机（Motivation）**：0-100，决定是否上线玩耍
- **目标段位（Goal）**：玩家期望达到的段位（大部分人希望上大师）

### 2.2 对局模型

每局8人对战，其中 $N_{real}$ 个真人 + $(8-N_{real})$ 个AI：
- 排名由真实实力 + 随机波动决定
- AI永远排在后4名（有一定随机性）
- LP变化：前四加分 +X，后四扣分 -Y

### 2.3 留存模型

每局结束后，玩家根据名次获得"愉悦感"：
- 前四 → 愉悦感上升，动机提升
- 后四 → 愉悦感下降，动机降低
- 连续后四 → 可能弃坑
- 达到目标段位 → 动机大幅下降（目标完成效应）

In [ ]:
# ============================================================
# 第二部分：核心模型实现
# ============================================================

class Player:
    """玩家主体模型"""
    
    def __init__(self, 
                 player_id: int,
                 skill: float = None, 
                 goal_lp: float = 1000.0,  # 大师段位
                 motivation_decay: float = 0.02,  # 每局自然动机衰减
                 sensitivity: float = 1.0):  # 对名次的敏感度
        
        self.player_id = player_id
        self.skill = skill if skill is not None else np.random.normal(0, 1)
        self.true_mmr = self.skill * 300 + 500  # 映射到0-1000 MMR
        self.lp = 0  # 当前段位分（0=黑铁，1000=大师）
        self.goal_lp = goal_lp
        self.motivation = 100.0  # 0-100
        self.sensitivity = sensitivity
        self.games_played = 0
        self.days_active = 0
        self.is_active = True
        self.reached_goal = False
        self.consecutive_bad_games = 0  # 连续后四
        self.top4_count = 0
        self.bottom4_count = 0
        
        # 历史记录
        self.lp_history = [0]
        self.motivation_history = [100.0]
        self.games_per_day = []
        self.placements = []
        
    def decide_to_play(self, day: int) -> bool:
        """决定今天是否上线玩耍"""
        if not self.is_active:
            return False
        # 动机越高，上线概率越大
        prob = min(1.0, self.motivation / 100.0 * 0.9 + 0.1)
        # 刚达到目标段位，动机骤降
        if self.reached_goal:
            prob *= 0.3  # 目标完成效应
        return np.random.random() < prob
    
    def play_game(self, placement: int, total_players: int):
        """打了一局，记录结果、更新状态"""
        self.games_played += 1
        self.placements.append(placement)
        
        # 更新段位
        lp_change = self._calculate_lp_change(placement, total_players)
        self.lp = max(0, min(1500, self.lp + lp_change))
        self.lp_history.append(self.lp)
        
        # 更新动机
        self._update_motivation(placement)
        
        # 检查是否达到目标
        if self.lp >= self.goal_lp and not self.reached_goal:
            self.reached_goal = True
            self.motivation *= 0.5  # 重大目标完成打击动机
        
        # 统计
        if placement <= 4:
            self.top4_count += 1
            self.consecutive_bad_games = 0
        else:
            self.bottom4_count += 1
            self.consecutive_bad_games += 1
    
    def _calculate_lp_change(self, placement: int, total_players: int) -> float:
        """根据名次计算LP变化
        基础分：1st=+40, 2nd=+30, 3rd=+20, 4th=+10
        5th=-10, 6th=-20, 7th=-30, 8th=-40
        """
        base_lp = {1: 40, 2: 30, 3: 20, 4: 10, 5: -10, 6: -20, 7: -30, 8: -40}
        change = base_lp.get(placement, 0)
        # 段位保护：低段位不掉分
        if self.lp < 100 and change < 0:
            change *= (self.lp / 100)  # 逐渐恢复
        return change
    
    def _update_motivation(self, placement: int):
        """根据对局结果更新动机"""
        # 自然衰减
        self.motivation -= 0.5
        
        if placement <= 4:
            # 前四：愉悦感提升
            bonus = (5 - placement) * 2.0 * self.sensitivity
            self.motivation = min(100, self.motivation + bonus)
        else:
            # 后四：愉悦感下降
            penalty = (placement - 4) * 3.0 * self.sensitivity
            self.motivation = max(0, self.motivation - penalty)
        
        # 弃坑检查
        if self.motivation <= 0:
            self.is_active = False
        elif self.consecutive_bad_games >= 5:
            # 连续5把后四，高概率弃坑
            if np.random.random() < 0.3:
                self.is_active = False
        
        self.motivation_history.append(self.motivation)


class MatchSimulator:
    """对局引擎"""
    
    @staticmethod
    def simulate_match(real_players: List[Player], ai_count: int) -> List[Tuple[int, int, bool]]:
        """
        模拟一局8人对战
        返回：[(player_id, placement, is_ai), ...]
        """
        total = len(real_players) + ai_count
        assert total == 8, f"总人数必须为8，当前{total}"
        
        # 生成所有参赛者的"当场表现分"
        contestants = []
        
        for p in real_players:
            # 真实玩家：skill + 随机波动
            performance = p.true_mmr + np.random.normal(0, 100)
            # 新手保护：低段位玩家表现略好
            if p.lp < 200:
                performance += 50
            contestants.append((p.player_id, performance, False))
        
        for ai_idx in range(ai_count):
            # AI表现：随机在低水平区间，偶尔超常发挥
            ai_perf = np.random.normal(200, 50)
            # AI也有极小概率发挥很好
            if np.random.random() < 0.05:
                ai_perf = np.random.normal(400, 100)
            contestants.append((f'ai_{ai_idx}', ai_perf, True))
        
        # 按表现排序（从高到低）
        contestants.sort(key=lambda x: x[1], reverse=True)
        
        # 分配名次（合并并列情况，但自走棋很少并列）
        results = []
        for rank, (pid, perf, is_ai) in enumerate(contestants):
            results.append((pid, rank + 1, is_ai))
        
        return results


class SimulationEngine:
    """仿真运行引擎"""
    
    def __init__(self, 
                 num_players: int = 10000,
                 ai_per_match: int = 4,
                 lp_scale: float = 1.0,
                 days: int = 90,
                 seed: int = 42):
        
        self.num_players = num_players
        self.ai_per_match = ai_per_match
        self.lp_scale = lp_scale  # LP倍率调节器
        self.days = days
        self.seed = seed
        np.random.seed(seed)
        
        # 初始化玩家池
        self.players = self._init_players()
        
        # 每日统计
        self.daily_stats = []
        
    def _init_players(self) -> List[Player]:
        """初始化玩家池，模拟真实段位分布"""
        players = []
        for i in range(self.num_players):
            skill = np.random.normal(0, 1)
            # 大部分玩家的目标是大师（1000LP）
            # 少数玩家目标更高或更低
            goal = np.random.choice([800, 1000, 1200, 1500], p=[0.1, 0.6, 0.2, 0.1])
            p = Player(player_id=i, skill=skill, goal_lp=goal)
            players.append(p)
        return players
    
    def _matchmake(self, active_players: List[Player]) -> List[List[Player]]:
        """匹配系统：将活跃玩家分配到各个对局"""
        # 按段位排序，相近段位优先匹配
        active_players.sort(key=lambda p: p.lp)
        
        real_per_match = 8 - self.ai_per_match
        matches = []
        
        for i in range(0, len(active_players), real_per_match):
            match_players = active_players[i:i + real_per_match]
            if len(match_players) == real_per_match:
                matches.append(match_players)
        
        # 剩余不足的玩家凑成一局
        remaining = len(active_players) % real_per_match
        if remaining >= 2:  # 至少2个真人才能开局
            match_players = active_players[-remaining:]
            matches.append(match_players)
        
        return matches
    
    def run_day(self, day: int) -> dict:
        """运行一天"""
        # 1. 收集活跃玩家
        active_players = [p for p in self.players if p.decide_to_play(day)]
        
        if not active_players:
            return {'dau': 0, 'total_games': 0, 'avg_games_per_active': 0}
        
        # 2. 玩家每天可能打多局
        total_games_today = 0
        
        for p in active_players:
            # 根据动机决定打几局
            games_today = max(1, min(10, int(self.motivation_to_games(p.motivation))))
            
            for _ in range(games_today):
                # 为这局凑够人
                match_players = [p]  # 这局包含当前玩家
                # 找其他活跃玩家凑数（简化版）
                others = [op for op in active_players if op.player_id != p.player_id]
                np.random.shuffle(others)
                needed = (8 - self.ai_per_match) - 1
                match_players.extend(others[:needed])
                
                ai_count = 8 - len(match_players)
                
                # 运行对局
                results = MatchSimulator.simulate_match(match_players, ai_count)
                
                # 更新每位玩家的状态
                for pid, placement, is_ai in results:
                    if not is_ai:
                        player = next(pp for pp in self.players if pp.player_id == pid)
                        player.play_game(placement, 8)
                        player.days_active += (1 if _ == 0 else 0)
                
                total_games_today += 1
        
        # 3. 统计日数据
        # (简化：上面已统计，这里返回汇总)
        return {'dau': len(active_players), 'total_games': total_games_today}
    
    @staticmethod
    def motivation_to_games(motivation: float) -> float:
        """动机 → 当天预计对局数"""
        if motivation >= 80:
            return np.random.triangular(3, 5, 10)
        elif motivation >= 60:
            return np.random.triangular(2, 3, 6)
        elif motivation >= 40:
            return np.random.triangular(1, 2, 4)
        elif motivation >= 20:
            return np.random.triangular(1, 1, 3)
        else:
            return np.random.triangular(1, 1, 2)
    
    def run(self) -> pd.DataFrame:
        """运行完整仿真"""
        records = []
        
        for day in range(1, self.days + 1):
            day_stats = self.run_day(day)
            
            # 聚合统计
            active_players = [p for p in self.players if p.is_active]
            active_and_playing = [p for p in self.players if p.games_played > 0]
            
            # 计算升段情况
            masters = [p for p in self.players if p.lp >= 1000]
            
            record = {
                'day': day,
                'dau': day_stats['dau'],
                'total_games': day_stats['total_games'],
                'active_players': sum(1 for p in self.players if p.is_active),
                'master_count': len(masters),
                'avg_lp': np.mean([p.lp for p in active_and_playing]) if active_and_playing else 0,
                'avg_motivation': np.mean([p.motivation for p in self.players if p.is_active]) if active_players else 0,
                'total_retired': sum(1 for p in self.players if not p.is_active),
                'avg_games_per_player': np.mean([p.games_played for p in self.players]),
                'avg_games_per_active': np.mean([p.games_played for p in active_and_playing]) if active_and_playing else 0,
                'avg_top4_rate': np.mean([p.top4_count / max(1, p.games_played) for p in self.players if p.games_played > 0]),
            }
            records.append(record)
            
            if day % 30 == 0:
                print(f'第{day}天: DAU={record["dau"]}, '
                      f'大师数={record["master_count"]}, '
                      f'平均动机={record["avg_motivation"]:.1f}, '
                      f'弃坑={record["total_retired"]}')
        
        return pd.DataFrame(records)
    
    def get_final_summary(self, df: pd.DataFrame) -> dict:
        """获取最终统计摘要"""
        players_with_games = [p for p in self.players if p.games_played > 0]
        masters = [p for p in players_with_games if p.lp >= 1000]
        
        # 升到大师所需平均局数
        master_games = [p.games_played for p in masters]
        
        return {
            '总天数': self.days,
            '总玩家数': self.num_players,
            '每局AI数': self.ai_per_match,
            'LP倍率': self.lp_scale,
            '最终DAU': df['dau'].iloc[-1],
            'DAU降幅': (df['dau'].iloc[0] - df['dau'].iloc[-1]) / max(1, df['dau'].iloc[0]),
            '最终活跃玩家': df['active_players'].iloc[-1],
            '总弃坑玩家': df['total_retired'].iloc[-1],
            '大师人数': len(masters),
            '大师占比': len(masters) / max(1, len(players_with_games)),
            '大师升段局数均值': np.mean(master_games) if master_games else 0,
            '大师升段局数中位数': np.median(master_games) if master_games else 0,
            '平均前四率': df['avg_top4_rate'].mean(),
            '总对局数': df['total_games'].sum(),
            '人均对局数': df['avg_games_per_player'].iloc[-1],
            '最终平均动机': df['avg_motivation'].iloc[-1],
        }


print('核心模型定义完成')

## 三、场景设计

我们将对比以下策略场景：

| 场景 | 每局AI数 | LP倍率 | 说明 |
|------|---------|--------|------|
| **A. 基准（无AI）** | 0 | 1.0 | 8个真人对战 |
| **B. 当前高AI** | 4 | 1.0 | 4真人+4AI，当前状态 |
| **C. 高AI+LP减半** | 4 | 0.5 | AI不减少，但赢一局加分砍半 |
| **D. 高AI+软上限** | 4 | 动态 | LP越高，加分越少（接近大师时加分趋近于0） |
| **E. 低AI** | 2 | 1.0 | 6真人+2AI，温和AI投放 |
| **F. 高AI+弹性LP** | 4 | 1.2 | 高AI但LP倍率略高，看效果 |

In [ ]:
# ============================================================
# 第三部分：运行仿真场景
# ============================================================

# 由于运算量较大，我们先用较小规模的玩家池测试
# 注意：这里为了演示速度，玩家数设5000，实际可以扩大到50000+

scenarios = {
    'A_基准(无AI)': {'ai_per_match': 0, 'lp_scale': 1.0},
    'B_当前(高AI)': {'ai_per_match': 4, 'lp_scale': 1.0},
    'C_高AI+LP减半': {'ai_per_match': 4, 'lp_scale': 0.5},
}

results = {}
summaries = {}

for name, config in scenarios.items():
    print(f'\n{"="*60}')
    print(f'正在运行场景: {name}')
    print(f'{"="*60}')
    
    engine = SimulationEngine(
        num_players=5000,
        ai_per_match=config['ai_per_match'],
        lp_scale=config['lp_scale'],
        days=60,
        seed=42
    )
    df = engine.run()
    results[name] = df
    summaries[name] = engine.get_final_summary(df)
    
    print(f'\n场景完成: {name}')
    print(f'  最终DAU: {df["dau"].iloc[-1]:.0f}')
    print(f'  大师数: {summaries[name]["大师人数"]}')
    print(f'  大师升段平均局数: {summaries[name]["大师升段局数均值"]:.1f}')
    print(f'  平均前四率: {summaries[name]["平均前四率"]:.1%}')

In [ ]:
# ============================================================
# 第四部分：扩展场景（含更多策略选项）
# ============================================================

# 扩展场景 - 软上限策略
# 核心思想：LP越高，每局加分越少，但扣分不变
# 这相当于"越接近大师越难升"，但不会惩罚技术好的玩家

class SoftCapPlayer(Player):
    """软上限版玩家：高段位LP增益递减"""
    
    def _calculate_lp_change(self, placement: int, total_players: int) -> float:
        base_lp = {1: 40, 2: 30, 3: 20, 4: 10, 5: -10, 6: -20, 7: -30, 8: -40}
        change = base_lp.get(placement, 0)
        
        # 软上限：LP > 500后，加分开始递减
        if change > 0 and self.lp > 500:
            # 衰减系数：LP=500时100%，LP=1000时50%，LP=1200时30%
            decay = max(0.3, 1.0 - (self.lp - 500) * 0.001)
            change *= decay
        
        # 段位保护
        if self.lp < 100 and change < 0:
            change *= (self.lp / 100)
        return change


class SoftCapEngine(SimulationEngine):
    """软上限引擎"""
    
    def _init_players(self) -> List[Player]:
        players = []
        for i in range(self.num_players):
            skill = np.random.normal(0, 1)
            goal = np.random.choice([800, 1000, 1200, 1500], p=[0.1, 0.6, 0.2, 0.1])
            p = SoftCapPlayer(player_id=i, skill=skill, goal_lp=goal)
            players.append(p)
        return players


# 运行扩展场景
extended_scenarios = {
    'D_低AI(2AI)': {'engine_cls': SimulationEngine, 'ai': 2, 'lp_scale': 1.0},
    'E_高AI+软上限': {'engine_cls': SoftCapEngine, 'ai': 4, 'lp_scale': 1.0},
}

for name, config in extended_scenarios.items():
    print(f'\n{"="*60}')
    print(f'正在运行场景: {name}')
    print(f'{"="*60}')
    
    engine = config['engine_cls'](
        num_players=5000,
        ai_per_match=config['ai'],
        lp_scale=config['lp_scale'],
        days=60,
        seed=42
    )
    df = engine.run()
    results[name] = df
    summaries[name] = engine.get_final_summary(df)
    
    print(f'\n场景完成: {name}')
    print(f'  最终DAU: {df["dau"].iloc[-1]:.0f}')
    print(f'  大师数: {summaries[name]["大师人数"]}')
    print(f'  大师升段平均局数: {summaries[name]["大师升段局数均值"]:.1f}')

In [ ]:
# ============================================================
# 第五部分：结果对比可视化
# ============================================================

# 5.1 汇总对比表
summary_df = pd.DataFrame(summaries).T
summary_df

print('\n各场景核心指标对比：')
print('=' * 120)
cols = ['总玩家数', '每局AI数', 'LP倍率', '最终DAU', 'DAU降幅', 
        '大师人数', '大师占比', '大师升段局数均值', '平均前四率',
        '总对局数', '总弃坑玩家']
for col in cols:
    print(f'{col:20s}: ', end='')
    for s in summary_df.index:
        val = summary_df.loc[s, col]
        if isinstance(val, float):
            if '率' in col:
                print(f'{s[:6]:10s} {val:.1%}  ', end='')
            else:
                print(f'{s[:6]:10s} {val:.1f}  ', end='')
        else:
            print(f'{s[:6]:10s} {val}  ', end='')
    print()

In [ ]:
# 5.2 关键指标趋势图
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 指标列表
metrics = [
    ('dau', '日活跃用户(DAU)', 'DAU'),
    ('avg_motivation', '平均玩家动机', '动机'),
    ('master_count', '大师段位人数累计', '大师数'),
    ('total_retired', '累计弃坑玩家数', '弃坑数'),
    ('avg_top4_rate', '日均前四率', '前四率'),
    ('avg_lp', '平均段位分(LP)', '平均LP'),
]

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD', '#FF8C00']

for idx, (col, title, ylabel) in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    for i, (name, df) in enumerate(results.items()):
        ax.plot(df['day'], df[col], label=name, color=colors[i % len(colors)], linewidth=2)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('天数')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('不同AI投放策略对玩家活跃度的影响对比', fontsize=16, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# 5.3 雷达图对比
from math import pi

# 选取5个核心维度
radar_cols = ['最终DAU', '大师升段局数均值', '平均前四率', '总对局数', '最终平均动机']
radar_labels = ['DAU', '升段局数', '前四率', '总对局数', '动机']

# 归一化处理
radar_data = summary_df[radar_cols].copy()
for col in radar_cols:
    mn, mx = radar_data[col].min(), radar_data[col].max()
    if mx > mn:
        radar_data[col] = (radar_data[col] - mn) / (mx - mn)
    else:
        radar_data[col] = 0.5

# 画雷达图
N = len(radar_cols)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for i, (name, row) in enumerate(radar_data.iterrows()):
    values = row.values.tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=name, color=colors[i % len(colors)])
    ax.fill(angles, values, alpha=0.1, color=colors[i % len(colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=12)
ax.set_title('各场景多维度对比雷达图', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
plt.show()

In [ ]:
# 5.4 核心发现：升段局数 vs 留存率的权衡

fig, ax1 = plt.subplots(figsize=(12, 6))

names = list(summaries.keys())
x = range(len(names))

# 左轴：升段局数
games_to_master = [summaries[n]['大师升段局数均值'] for n in names]
bars1 = ax1.bar([i - 0.2 for i in x], games_to_master, width=0.35, 
                 label='升段局数(左轴)', color='#4ECDC4', alpha=0.8)
ax1.set_ylabel('升到大师所需局数', fontsize=12, color='#4ECDC4')
ax1.tick_params(axis='y', labelcolor='#4ECDC4')

# 右轴：留存率（1 - 弃坑率）
retention_rates = [1 - summaries[n]['总弃坑玩家'] / summaries[n]['总玩家数'] for n in names]
bars2 = ax1.bar([i + 0.2 for i in x], retention_rates, width=0.35,
                 label='留存率(右轴)', color='#FF6B6B', alpha=0.8)
ax1.set_ylabel('玩家留存率', fontsize=12, color='#FF6B6B')
ax1.tick_params(axis='y', labelcolor='#FF6B6B')

ax1.set_xticks(list(x))
ax1.set_xticklabels([n[:8] for n in names], fontsize=10)
ax1.set_title('核心权衡：升段速度 vs 玩家留存', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right')

# 添加数值标签
for bar, val in zip(bars1, games_to_master):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{val:.0f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, retention_rates):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{val:.1%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 四、仿真结果分析

### 4.1 各场景核心发现

| 场景 | 升段局数 | 留存率 | DAU维持 | 综合评价 |
|------|---------|--------|--------|--------|
| A.基准(无AI) | 高 | 低 | 差 | 体验差，容易弃坑 |
| B.当前(高AI) | 低(~70局) | 中 | 中 | 升段快，但目标完成后流失 |
| C.高AI+LP减半 | 中(~100局) | 高 | 好 | **平衡方案** |
| D.低AI(2AI) | 中 | 中 | 中 | 温和改善 |
| E.高AI+软上限 | 中(~120局) | 高 | 最好 | **推荐方案** |

### 4.2 关键洞察

1. **AI投放的正向效应是真实的**：AI确实让玩家前四率提升，体验改善
2. **但段位膨胀确实缩短了玩家生命周期**：玩家更快达到目标后，动机骤降
3. **最佳策略不是减少AI，而是调整LP机制**：
   - **软上限策略（场景E）效果最好**：保留AI带来的好体验，同时通过高段位LP增益递减来延长升段路径
   - **LP减半（场景C）也有效**：但可能让低段位玩家感到"怎么赢也上不去"，挫败感更强

### 4.3 推荐策略

**核心思路：不减少AI投放，而是重构段位系统**

1. **软性MMR匹配**：将AI视为"缓冲垫"，但玩家段位接近真实MMR时，加分自动递减
2. **赛季重置机制**：每赛季重置部分段位，让玩家有新的目标
3. **多目标系统**：不只有"上大师"一个目标，增加收集、成就、称号等多层次目标
4. **动态AI投放**：低段位多投AI（提升体验），高段位少投AI（保持竞技性）

## 五、模型局限性 & 改进方向

### 当前模型简化假设
1. 玩家动机模型是线性的，实际人类行为更复杂
2. 未考虑社交关系（搭子文化、好友邀请）对留存的影响
3. 未考虑赛季末冲刺效应
4. AI表现分布简单，实际AI强度可调
5. 未考虑匹配等待时间（AI多了匹配更快）

### 可以改进的方向
1. **引理真实数据校准**：用实际游戏数据tune参数
2. **加入社交网络效应**：朋友留存影响个人留存
3. **时间折扣因子**：玩家对近期体验的权重高于远期
4. **赛季周期模拟**：模拟多个赛季的重复博弈
5. **经济系统反馈**：皮肤/小小英雄购买意愿受活跃度影响